# PhysIR — Stage 6: Seed Replicates @ 256/200 · Colab (single flow)

**Purpose:** seed-robustness for the headline physics contrast. Re-trains the two
load-bearing arms — `abl_baseline` (physics OFF) and `abl_full` (phase 0.20 + retinex 0.18)
— at **two extra seeds (3 and 5**; 7 reserved as optional third; **primary thesis seed = 42**,
data split ALWAYS seed-42). Regime matches the Kaggle Set runs exactly: crop 256,
**effective batch 8** (single-GPU batch 8 ≡ Kaggle's 4×2 DDP — identical mean gradient),
lr 2.8e-4 cosine, EMA, 200 ep, ep150 snapshot, save_freq 2 (atomic).

**Loop order is PAIR-MAJOR** — s1337-baseline → s1337-full → s2024-baseline → s2024-full — so a budget
cut always leaves complete, comparable pairs. **Budget honesty: 4 runs ≈ 150–170 CU on A100
(~35–45 CU each); with ~90 CU expect the first pair + change. Everything is
Drive-persistent + resume-safe: re-run this notebook after the monthly CU reset (or finish
remaining seeds on Kaggle via the set notebooks' SEED knob) and it continues exactly where
it stopped.** Use the **A100** runtime (H100's premium buys nothing at 256/batch-8).

Eval per finished run: LOLBlur test at {ep150, best, last}, EMA weights → per-image CSVs
(`per_image_<arm>_s<seed>_<ckpt>.csv`) for cross-seed Wilcoxon/variance analysis offline.

## 1. Drive (persistent output)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

OUT_ROOT = '/content/drive/MyDrive/PhysIR/stage6_replicates'
CKPT_DIR = os.path.join(OUT_ROOT, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_ROOT, '.metrics'), exist_ok=True)
print('Persistent output →', OUT_ROOT)

## 2. Install

In [ ]:
!pip install -q lpips pytorch-msssim einops ptflops pyiqa kaggle

## 3. Data + code + guard (Kaggle pulls; junk-cleaned)

In [ ]:
import re
from google.colab import userdata
import os, sys, glob

# ── Kaggle auth via Colab Secret COLAB_TOKEN (your preferred method) ───────────
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(userdata.get("COLAB_TOKEN"))
# also write kaggle.json if the secret is JSON creds (harmless fallback for the CLI)
try:
    _tok = userdata.get("COLAB_TOKEN")
    if _tok.lstrip().startswith("{"):
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
            f.write(_tok)
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
except Exception:
    pass

# ── EDIT: dataset slugs (owner/dataset) ───────────────────────────────────────
KAGGLE_CODE     = 'jeffreyboahen/physir-code'
KAGGLE_LOLBLUR  = 'jeffreyboahen/lolblur-dataset'
KAGGLE_LOLV2    = ''   # not needed for replicates
KAGGLE_REALBLUR = ''   # not needed for replicates
KAGGLE_LSRW     = ''   # not needed for replicates
# ──────────────────────────────────────────────────────────────────────────────

# ABSOLUTE dest (we chdir to Drive later, so relative ./data would break)
DATA = '/content/data'
os.makedirs(DATA, exist_ok=True)

def pull(slug, sub):
    if not slug:
        print(f'  (skip {sub})'); return
    d = os.path.join(DATA, sub)
    if os.path.isdir(d) and os.listdir(d):
        print(f'  {sub}: already present'); return
    os.makedirs(d, exist_ok=True)
    print(f'  downloading {slug} -> {sub} ...')
    os.system(f'kaggle datasets download -d {slug} --unzip -p {d} -q')

pull(KAGGLE_CODE,     'code')
pull(KAGGLE_LOLBLUR,  'lolblur-dataset')
pull(KAGGLE_LOLV2,    'lolv2-dataset')
pull(KAGGLE_REALBLUR, 'real-lolblur')
pull(KAGGLE_LSRW,     'lsrw')

# ── locate roots (datasets often nest one level deep) ─────────────────────────
def find_root(base, marker, depth=3):
    if os.path.exists(os.path.join(base, marker)): return base
    for dd in range(1, depth+1):
        hits = sorted(glob.glob(os.path.join(base, *(['*']*dd), marker)))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep)
    return None

CODE_ROOT = find_root(os.path.join(DATA, 'code'), 'train.py')
assert CODE_ROOT, 'train.py not found under the code dataset'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# ── GUARD: pulled code MUST contain the EMA + resume fixes ────────────────────
src = open(os.path.join(CODE_ROOT, 'train.py')).read()
missing = [n for n in ('class ModelEMA', 'restore_phys_state', "ckpt['ema']",
                       'snapshot_epochs', 'Seeded RNGs', '_atomic_save') if n not in src]
if missing:
    raise SystemExit(f'STALE physir-code - missing {missing}. '
                     f'Re-upload the physir-code Kaggle dataset with the current train.py.')
# ── ORDERING GUARD: the test loader must not shuffle, or per-image rows become a random
# permutation and every paired statistic built on them is invalid (the 2026-07-30 bug).
_dsrc = open(os.path.join(CODE_ROOT, 'data', 'dataset_reader', 'dataset_LOLBlur.py')).read()
_shuf = [d for d in re.findall(r'DataLoader\([^)]*\)', _dsrc, re.S)
         if 'shuffle=True' in d.replace(' ', '')]
assert not _shuf, ('STALE physir-code: test DataLoader still has shuffle=True — '
                   're-upload physir-code with the shuffle=False fix.')
print('✅ ordering guard passed (test loader is not shuffled)')
print(f'\n[OK] CODE_ROOT = {CODE_ROOT}  (EMA + resume + LPIPSLoss present)')

LOLBLUR_ROOT = find_root(os.path.join(DATA, 'lolblur-dataset'),
                         os.path.join('train', 'high_sharp_scaled'))
assert LOLBLUR_ROOT, 'LOLBlur train not found'
print(f'LOLBLUR_ROOT = {LOLBLUR_ROOT}')
print('  has test/ :', os.path.isdir(os.path.join(LOLBLUR_ROOT, 'test')))

# strip macOS AppleDouble junk ('._*.png' crashes PIL — extension filters can't catch it)
!find /content/data \( -name '._*' -o -name '.DS_Store' \) -type f -delete 2>/dev/null


## 4. Split (90/10, seed 42 — NEVER changes with training seed)

In [ ]:
import random
from pathlib import Path

SUBS      = ['low_blur_noise', 'high_sharp_scaled']
TRAIN_SRC = Path(LOLBLUR_ROOT) / 'train'
SPLIT     = Path('/content/LOLBlur_split')

random.seed(42)
clips = sorted([d.name for d in (TRAIN_SRC / 'low_blur_noise').iterdir() if d.is_dir()])
n_val = max(1, int(len(clips) * 0.1))
val_clips = set(random.sample(clips, n_val))
train_clips = [c for c in clips if c not in val_clips]
print(f'clips — train {len(train_clips)}, val {len(val_clips)}')

for name, cl in [('physir_train', train_clips), ('physir_val', val_clips)]:
    for sub in SUBS:
        base = SPLIT / name / sub; base.mkdir(parents=True, exist_ok=True)
        for c in cl:
            dst = base / c
            if not dst.exists():
                dst.symlink_to((TRAIN_SRC / sub / c).resolve())

PHYSIR_TRAIN = str(SPLIT / 'physir_train')
PHYSIR_VAL   = str(SPLIT / 'physir_val')
print('train:', PHYSIR_TRAIN, '\nval:  ', PHYSIR_VAL)

## 5. Runs — 2 seeds × 2 arms, pair-major

In [ ]:
import yaml, itertools
EPOCHS   = 200
SNAPSHOT = 150
SEEDS    = [1337, 2024]  # extra seeds -> 3 TOTAL with primary 42 (odd-count convention: 3/5/7 total).
ARMS = {
  'abl_baseline': dict(phase=0.0,  retinex=0.0),
  'abl_full':     dict(phase=0.20, retinex=0.18),
}
# PAIR-MAJOR order: complete each seed's {baseline, full} pair before the next seed
RUNS = [(f'{arm}_s{seed}', arm, seed, sp) for seed in SEEDS for arm, sp in ARMS.items()]

def build_cfg(name, sp, seed):
    phys_on = sp['phase'] > 0 or sp['retinex'] > 0
    cfg = {
      'seed': seed,
      'datasets': {'name': 'LOLBlur',
        'train': {'low_path': f'{PHYSIR_TRAIN}/low_blur_noise', 'high_path': f'{PHYSIR_TRAIN}/high_sharp_scaled',
                  'crop_size': 256, 'batch_size': 8, 'n_workers': 8, 'use_flips': True},
        'val': {'test_path': PHYSIR_VAL, 'batch_size_test': 1, 'verbose': False, 'n_workers': 4}},
      'network': {'name': 'DarkIR', 'img_channels': 3, 'width': 32, 'middle_blk_num_enc': 2,
                  'middle_blk_num_dec': 2, 'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1],
                  'dilations': [1, 4, 9], 'extra_depth_wise': True},
      'save': {'pretrained': None, 'path': f'{CKPT_DIR}/{name}_latest.pt', 'best': f'{CKPT_DIR}/{name}_best.pt',
               'save_freq': 2, 'eval_freq': 10, 'perceptual_eval_epochs': [],
               'snapshot_epochs': [SNAPSHOT]},
      'pixel_criterion': 'l1', 'perceptual': False,
      'edge': True, 'edge_weight': 0.05, 'edge_criterion': 'l2', 'edge_reduction': 'mean',
      'frequency': False,
      'enhance': True, 'enhance_weight': 0.01, 'enhance_criterion': 'l1', 'enhance_reduction': 'mean',
      'physics_losses': {'enabled': phys_on, 'phase2_start': 9999, 'phase3_start': 9999, 'phase4_start': 1,
                         'retinex': sp['retinex'] > 0, 'retinex_weight': sp['retinex'],
                         'blur': False, 'blur_weight': 0.0,
                         'phase': sp['phase'] > 0, 'phase_weight': sp['phase']},
      'train': {'epochs': EPOCHS, 'lr_initial': 0.00028, 'weight_decay': 0.001,
                'lr_scheme': 'CosineAnnealing', 'eta_min': 1e-6, 'grad_clip': 1.0,
                'ema': {'enabled': True, 'decay': 0.9999, 'warmup': 2000}},
      'wandb': {'init': False, 'name': name}, 'Resize': False,
    }
    p = f'{OUT_ROOT}/{name}.yml'
    yaml.dump(cfg, open(p, 'w'), default_flow_style=False)
    return p
print('runs (pair-major):', [r[0] for r in RUNS], '| ~35-45 CU each on A100')

## 6. TRAIN loop (idempotent; resumes across sessions/CU refills)

In [ ]:
import csv, subprocess, torch, sys
os.chdir(OUT_ROOT)   # train.py writes .metrics/<run> under Drive

def done_ep(name):
    ck = os.path.join(CKPT_DIR, f'{name}_latest.pt')
    if os.path.isfile(ck):
        try:
            return int(torch.load(ck, map_location='cpu', weights_only=False).get('epoch', 0))
        except Exception as e:
            print(f'  ({name}_latest.pt unreadable: {e} — log fallback)')
    lg = os.path.join(OUT_ROOT, '.metrics', name, 'training_log.csv')
    if not os.path.isfile(lg): return 0
    last = 0
    try:
        for r in csv.DictReader(open(lg)): last = max(last, int(r['epoch']))
    except Exception: pass
    return last

for name, arm, seed, sp in RUNS:
    d = done_ep(name)
    if d >= EPOCHS:
        print(f'✅ {name}: {d}/{EPOCHS} — skip'); continue
    p = build_cfg(name, sp, seed)
    print(f'\n===== TRAIN {name} (seed {seed}) {"(resume ep %d)" % d if d else "(scratch)"} =====')
    r = subprocess.run([sys.executable, f'{CODE_ROOT}/train.py', '-p', p])
    print(f'===== {name} exit {r.returncode} | now {done_ep(name)}/{EPOCHS} =====')
print('\nTRAIN loop done —', {n: f'{done_ep(n)}/{EPOCHS}' for n, *_ in RUNS})

## 7. EVAL — {ep150, best, last} per finished run

In [ ]:
import numpy as np, csv as _csv
import train as T
from data import create_test_data
device = 'cuda'
opt_test = {'name': 'LOLBlur', 'val': {'test_path': os.path.join(LOLBLUR_ROOT, 'test'), 'batch_size_test': 1},
            'train': {'n_workers': 4, 'verbose': True}}
tl_raw, _ = create_test_data(rank=0, world_size=1, opt=opt_test)
tl = tl_raw if not isinstance(tl_raw, dict) else list(tl_raw.values())[0]['loader']
# clip-qualified image keys (LOLBlur test basenames repeat across its 30 clips), written to
# every row so a future ordering change is DETECTABLE instead of silent.
_ds = (tl if 'tl' in dir() else LOADER).dataset
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p)) for p in _ds.imgs_low]
assert len(set(FNAMES)) == len(FNAMES), 'image keys are not unique'
lpips_fn = T._LPIPS(net='vgg', verbose=False).to(device)
PERIMG = os.path.join(OUT_ROOT, '.metrics', 'per_image'); os.makedirs(PERIMG, exist_ok=True)
NET = {'name': 'DarkIR', 'img_channels': 3, 'width': 32, 'middle_blk_num_enc': 2, 'middle_blk_num_dec': 2,
       'enc_blk_nums': [1, 2, 3], 'dec_blk_nums': [3, 1, 1], 'dilations': [1, 4, 9], 'extra_depth_wise': True}

@torch.no_grad()
def eval_test(model, out_csv):
    rows = []
    for i, (high, low) in enumerate(tl):
        high, low = high.to(device), low.to(device)
        pred = model(low, side_loss=False).clamp(0, 1)
        _, _, H, W = high.shape; pred = pred[:, :, :H, :W]
        mse = torch.mean((pred - high) ** 2).item()
        psnr = 20 * np.log10(1.0 / (np.sqrt(mse) + 1e-8))
        T._ssim_metric.to(device); ssim = T._ssim_metric(pred, high).item()
        lp = lpips_fn(pred * 2 - 1, high * 2 - 1).mean().item()
        rows.append((i, psnr, ssim, lp))
    with open(out_csv, 'w', newline='') as f:
        w = _csv.writer(f); w.writerow(['idx', 'psnr', 'ssim', 'lpips']); w.writerows(rows)
    a = np.array([r[1:] for r in rows])
    return a.mean(0)

for name, arm, seed, sp in RUNS:
    if done_ep(name) < EPOCHS:
        print(f'⏳ {name}: training incomplete — skip eval'); continue
    for tag, ck in [('ep150', f'{CKPT_DIR}/{name}_latest_ep{SNAPSHOT}.pt'),
                    ('best',  f'{CKPT_DIR}/{name}_best.pt'),
                    ('last',  f'{CKPT_DIR}/{name}_latest.pt')]:
        out = os.path.join(PERIMG, f'per_image_{name}_{tag}.csv')
        if os.path.exists(out): print(f'✅ {name}[{tag}] exists'); continue
        if not os.path.isfile(ck): print(f'⚠️ {name}[{tag}] ckpt missing'); continue
        c = torch.load(ck, map_location='cpu', weights_only=False)
        m = T.build_model(NET).to(device)
        st = c.get('ema') or c['model']
        m.load_state_dict({k.replace('module.', ''): v for k, v in st.items()}, strict=False); m.eval()
        ps, ss, lp = eval_test(m, out)
        print(f'{name}[{tag}] PSNR={ps:.4f} SSIM={ss:.4f} LPIPS={lp:.4f}')
        del m; torch.cuda.empty_cache()
print('\nEVAL done. CSVs in', PERIMG)

## 8. Leaderboard + cross-seed spread

In [ ]:
import pandas as pd, glob as _g
rows = []
for f in sorted(_g.glob(os.path.join(PERIMG, 'per_image_*.csv'))):
    df = pd.read_csv(f)
    base = os.path.basename(f)[len('per_image_'):-len('.csv')]
    run, tag = base.rsplit('_', 1)
    rows.append({'run': run, 'ckpt': tag, 'psnr': df.psnr.mean(), 'ssim': df.ssim.mean(), 'lpips': df.lpips.mean()})
if rows:
    b = pd.DataFrame(rows)
    print(b.to_string(index=False))
    last = b[b.ckpt == 'last']
    for arm in ('abl_baseline', 'abl_full'):
        sub = last[last.run.str.startswith(arm)]
        if len(sub) > 1:
            print(f'\n{arm} cross-seed (last): PSNR {sub.psnr.mean():.3f} ± {sub.psnr.max()-sub.psnr.min():.3f} (range)')
    print('\nCombine offline with the seed-42 Set-1/Set-2 CSVs for the full seed table;')
    print('physics delta per seed = full_sN − baseline_sN (paired Wilcoxon per seed).')
else:
    print('no eval CSVs yet')